In [ ]:
"""The analyses of the EMIT paper, in one file.

Each part below is one analysis.  It computes its measures, prints them, and draws every
plot as a file of its own (``<out>/expression_within_spot_r.pdf`` and ``.png``, and so on).
The values are also written to ``<out>/analyses.csv``.

    python reproduce/analyses.py                         # synthetic data, about a minute
    python reproduce/analyses.py --analyses expression cell_types
    python reproduce/analyses.py --section colon.npz --compare Thor=thor.npz STIE=stie.npz
    python reproduce/analyses.py --analyses real_section --spaceranger outs/ --nuclei nuclei.npz
    python reproduce/analyses.py --analyses visium_hd --scene colon_hd.npz

    expression    expression of single cells inside and between spots
    cell_types    which cell is which inside a spot
    all_cells     one map of all cells: tissue domains, cell types, spatially organized genes
    real_section  a real Visium section: every cell mapped, and the measurement conserved
    visium_hd     single cells from Visium HD bins
    interface     differential expression where two cell types share a spot; gene programmes

``expression``, ``cell_types``, ``all_cells`` and ``interface`` use a section measured at
single-cell resolution, from which Visium-like spots are simulated: EMIT sees the spots and
the features of the cells, and its output is scored against the cells it was never shown.
``real_section`` uses a real Visium section, and ``visium_hd`` a tissue window in which the
owner cell of every transcript is known.  The file formats are described in
reproduce/README.md.
"""
from __future__ import annotations

import argparse
import os
import re
import sys
import textwrap
from contextlib import contextmanager

import numpy as np
from scipy import sparse
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial import cKDTree
from scipy.spatial.distance import squareform
from scipy.stats import rankdata, spearmanr

import emit
from emit._utils import group_mean, lognorm
from emit.hd import assign_bins
from emit.metrics import conservation_error, relative_distance, within_spot_r
from emit.simulate import toy_section, visium_spots

RESULTS = []  # one row per analysis, group of cells, method and measure


def report(analysis: str, title: str, table: dict) -> None:
    """Print ``{method: {measure: value}}`` and keep it for the output file."""
    measures = list(next(iter(table.values())))
    print(f"\n  {title}")
    print(f"    {'':<15s}" + "".join(f"{m:>30s}" for m in measures))
    for method, row in table.items():
        print(f"    {method:<15s}" + "".join(f"{row[m]:>30.4g}" for m in measures))
        RESULTS.extend({"analysis": analysis, "group": title, "method": method, "measure": m,
                        "value": float(row[m])} for m in measures)


def column(table: dict, measure: str, methods=None) -> dict:
    """One measure of a table, as ``{method: value}``."""
    return {name: row[measure] for name, row in table.items() if methods is None or name in methods}


# =============================================================================================
# Plotting: every plot is a file of its own, written to <out>/<name>.pdf and <name>.png
# =============================================================================================
PLOTS = {"dir": None, "show": False}  # set by main(); without a directory nothing is drawn
COLORS = {"EMIT": "#4c72b0", "Thor": "#dd8452", "STIE": "#55a868", "Interpolation": "#8c8c8c",
          "Spot mean": "#b0b0b0", "Nearest spot": "#b0b0b0", "EMIT-HD": "#4c72b0",
          "Radius 10 um": "#dd8452", "Oracle": "#333333"}
PALETTE = ["#4e79a7", "#f28e2b", "#e15759", "#59a14f", "#b07aa1", "#76b7b2", "#edc948", "#9c755f",
           "#ff9da7", "#bab0ac", "#86bcb6", "#d37295"]
STYLE = {"font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7, "xtick.labelsize": 6.5,
         "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.spines.top": False,
         "axes.spines.right": False, "pdf.fonttype": 42, "ps.fonttype": 42}


@contextmanager
def plot(name: str, ncols: int = 1, width: float = 2.6, height: float = 2.2):
    """One plot: yields its axes and saves it under ``name`` when the block ends."""
    import matplotlib.pyplot as plt

    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, ncols, figsize=(width * ncols, height), squeeze=False,
                                 layout="constrained")
        yield fig, list(axes[0])
        os.makedirs(PLOTS["dir"], exist_ok=True)
        path = os.path.join(PLOTS["dir"], name)
        fig.savefig(path + ".pdf")
        fig.savefig(path + ".png", dpi=300)
        if not PLOTS["show"]:
            plt.close(fig)
    print(f"    plot: {name}")


def draw_scores(name: str, values: dict, xlabel: str, reference: dict | None = None) -> None:
    """One dot per method on a common axis; ``reference`` adds labelled vertical marks."""
    if not PLOTS["dir"]:
        return
    with plot(name, width=3.2, height=0.8 + 0.27 * len(values)) as (_, (ax,)):
        for i, (method, value) in enumerate(values.items()):
            ax.axhline(i, color="#e3e6ea", lw=1.0, zorder=1)
            ax.plot(value, i, "o", ms=7 if method == "EMIT" else 5.5, mec="white", mew=0.5,
                    color=COLORS.get(method, "#777777"), zorder=3)
            ax.annotate(f"{value:.3g}", (value, i), xytext=(6, 0), textcoords="offset points",
                        va="center", fontsize=6)
        for label, x in (reference or {}).items():
            ax.axvline(x, color="#333333", lw=0.8, ls="--", zorder=2)
            ax.annotate(label, (x, -0.6), ha="center", va="bottom", fontsize=5.5,
                        annotation_clip=False)
        ax.set_yticks(range(len(values)))
        ax.set_yticklabels(list(values))
        ax.set_ylim(len(values) - 0.5, -0.6)
        ax.set_xlabel(textwrap.fill(xlabel, 48))
        ax.margins(x=0.25)
        ax.locator_params(axis="x", nbins=5)


def draw_maps(name: str, maps: dict, label: str = "", cmap="viridis", vmin=None, vmax=None,
              notes: dict | None = None, colours=None) -> None:
    """One quantity mapped once per entry of ``maps`` (``{title: (xy, values)}``).

    ``colours`` is a list of colours for integer values, in which case no bar is drawn.
    """
    if not PLOTS["dir"]:
        return
    from matplotlib.colors import ListedColormap

    if colours is not None:
        cmap, vmin, vmax = ListedColormap(colours), -0.5, len(colours) - 0.5
    with plot(name, ncols=len(maps), width=1.9, height=2.0) as (fig, axes):
        for ax, (title, (xy, values)) in zip(axes, maps.items()):
            size = float(np.clip(4e4 / max(len(xy), 1), 0.05, 6.0))
            image = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=size, linewidths=0, cmap=cmap,
                               vmin=vmin, vmax=vmax, rasterized=True)
            ax.set_aspect("equal")
            ax.invert_yaxis()  # image orientation
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_title(title)
            ax.set_xlabel((notes or {}).get(title, ""), fontsize=6)
        if colours is None:
            fig.colorbar(image, ax=axes, shrink=0.8, label=label)


def draw_scatters(name: str, scatters: dict, xlabel: str, ylabel: str, diagonal: bool = True,
                  notes: dict | None = None) -> None:
    """One scatter per entry of ``scatters`` (``{title: (x, y)}``)."""
    if not PLOTS["dir"] or not scatters:
        return
    with plot(name, ncols=len(scatters), width=2.1, height=2.1) as (_, axes):
        for k, (ax, (title, (x, y))) in enumerate(zip(axes, scatters.items())):
            if diagonal:
                lo = float(min(np.nanmin(x), np.nanmin(y)))
                hi = float(max(np.nanmax(x), np.nanmax(y)))
                ax.plot([lo, hi], [lo, hi], color="#bbbbbb", lw=0.6)
            else:
                ax.axhline(0, color="#bbbbbb", lw=0.5)
                ax.axvline(0, color="#bbbbbb", lw=0.5)
            ax.scatter(x, y, s=3, color=COLORS.get(title, "#777777"), alpha=0.5, linewidths=0,
                       rasterized=True)
            ax.set_title(title + (f", {notes[title]}" if notes and title in notes else ""))
            ax.set_xlabel(xlabel)
            if k == 0:
                ax.set_ylabel(ylabel)


# =============================================================================================
# Inputs: a benchmark section, the EMIT output for it, and the outputs of other methods
# =============================================================================================
MIN_COUNTS = 10       # cells with fewer measured counts are not scored
MAX_DIST_UM = 60.0    # cells between spots are scored up to this distance from a spot centre


def load_section(path: str, features: str | None = None):
    """Simulated Visium spots of a section, and the measured cells they were made from."""
    if path == "toy":
        return toy_section(seed=0)
    with np.load(path, allow_pickle=True) as z:
        counts = np.asarray(z["counts"], np.float32)
        spots = visium_spots(counts, z["cell_xy"], float(z["px_um"]))
        sample = emit.Sample(
            counts=spots["counts"], genes=z["genes"], spot_xy=spots["spot_xy"],
            cell_xy=z["cell_xy"], morph=z["morph"], px_um=float(z["px_um"]),
            spot_diameter_px=spots["spot_diameter_px"], cell_spot=spots["cell_spot"],
            morph_names=z["morph_names"] if "morph_names" in z.files else None,
            name=os.path.splitext(os.path.basename(path))[0])
        cell_type = np.asarray(z["cell_type"], np.int64) if "cell_type" in z.files else None
    if features:
        sample.image_features = emit.read_image_features(features, sample.n_cells)
    if cell_type is None:  # evaluation-only types: clusters of the measured expression
        from sklearn.cluster import MiniBatchKMeans
        from sklearn.decomposition import TruncatedSVD

        svd = TruncatedSVD(min(30, counts.shape[1] - 1), random_state=0)
        kmeans = MiniBatchKMeans(10, random_state=0, n_init=5, batch_size=8192)
        cell_type = kmeans.fit_predict(svd.fit_transform(lognorm(counts)))
    return sample, {"counts": counts, "cell_type": cell_type}


def emit_output(sample, args):
    """Counts of every cell from EMIT, fitted once and kept in ``args.out``.

    Cells inside spots hold an allocation of the spot counts; cells between spots hold the
    prediction of the network.
    """
    os.makedirs(args.out, exist_ok=True)
    cache = os.path.join(args.out, f"{sample.name}_emit.npz")
    if os.path.exists(cache):
        with np.load(cache) as z:
            return z["expression"]
    emit.setup_logging()
    rounds = args.rounds or (8 if sample.name == "toy" else 20)
    result = emit.fit(sample, rounds=rounds, seed=args.seed, device=args.device, monitor=False)
    np.savez(cache, expression=result.expression, allocated=result.allocated)
    return result.expression


def load_compared(specs, sample) -> dict:
    """Outputs of other methods (``NAME=FILE``) as ``{name: (expression, valid)}``."""
    out = {}
    for spec in specs:
        name, _, path = spec.partition("=")
        with np.load(path, allow_pickle=True) as z:
            x = np.asarray(z["expression"], np.float32)
            valid = np.asarray(z["valid"], bool) if "valid" in z.files else np.ones(len(x), bool)
        if x.shape != (sample.n_cells, sample.n_genes):
            raise ValueError(f"{path}: expected {sample.n_cells} cells x {sample.n_genes} genes")
        out[name] = (x, valid)
    return out


# ---- shared definitions -----------------------------------------------------------------------
def interpolation(sample) -> np.ndarray:
    """Image-free baseline: the profile of its spot for a cell inside a spot, and the
    inverse-distance-weighted mean of the three nearest spots for a cell between spots."""
    profile = sample.counts / np.clip(sample.counts.sum(1, keepdims=True), 1e-12, None)
    dist, near = cKDTree(sample.spot_xy).query(sample.cell_xy, k=3)
    weight = 1.0 / np.clip(dist, 1e-6, None)
    weight /= weight.sum(1, keepdims=True)
    out = np.empty((sample.n_cells, sample.n_genes), np.float32)
    for a in range(0, sample.n_cells, 50_000):
        rows = slice(a, a + 50_000)
        out[rows] = sum(weight[rows, k:k + 1] * profile[near[rows, k]] for k in range(3))
    out[sample.inside] = profile[sample.cell_spot[sample.inside]]
    return out


def scored_cells(sample, truth, valid=()) -> np.ndarray:
    """Cells that enter the all-cell scores: enough measured counts, inside a spot or close
    to one, and returned by every compared method."""
    dist = cKDTree(sample.spot_xy).query(sample.cell_xy, k=1)[0] * sample.px_um
    keep = (truth["counts"].sum(1) >= MIN_COUNTS) & (truth["cell_type"] >= 0)
    keep &= sample.inside | (dist <= MAX_DIST_UM)
    for mask in valid:
        keep &= mask
    return keep


def measured_genes(genes) -> np.ndarray:
    """Mask of the scored genes: control probes of the panel are left out."""
    control = re.compile(r"(?i)^(neg|blank|unassigned|antisense|deprecated|intergenic|control)")
    return np.array([not control.match(str(g)) for g in genes])


def gene_index(genes, name: str | None, expression) -> int:
    """Column of the gene to map: the named one if it is present, else the most variable."""
    genes = [str(g) for g in genes]
    if name in genes:
        return genes.index(name)
    return int(np.argmax(lognorm(expression).var(0)))


def unit(x) -> np.ndarray:
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def clr(x, eps: float = 1e-6) -> np.ndarray:
    """Centred log-ratio of each row, on compositions."""
    x = np.clip(np.asarray(x, np.float64), 0, None)
    logc = np.log(x / np.clip(x.sum(1, keepdims=True), 1e-12, None) + eps)
    return logc - logc.mean(1, keepdims=True)


def type_centroids(counts, cell_type, n_types: int) -> np.ndarray:
    """Unit-length CLR profile of the mean measured counts of each cell type."""
    mean = np.zeros((n_types, counts.shape[1]))
    for k in range(n_types):
        if (cell_type == k).any():
            mean[k] = counts[cell_type == k].mean(0)
    return unit(clr(mean))


def read_out_types(expression, centroids, chunk: int = 100_000) -> np.ndarray:
    """Cell type read from an expression profile: the nearest measured type centroid.
    The same rule is applied to every method."""
    out = np.empty(len(expression), np.int64)
    for a in range(0, len(expression), chunk):
        out[a:a + chunk] = np.argmax(unit(clr(expression[a:a + chunk])) @ centroids.T, 1)
    return out


def pearson(a, b) -> float:
    a, b = a - a.mean(), b - b.mean()
    den = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / den) if den > 0 else 0.0


def neighbour_weights(xy_um, k: int):
    """Indices of the ``k`` nearest cells of every cell, and the matrix that averages over them."""
    nb = cKDTree(xy_um).query(xy_um, k=k + 1)[1][:, 1:]
    n = len(nb)
    rows = np.repeat(np.arange(n), k)
    return nb, sparse.csr_matrix((np.full(n * k, 1.0 / k), (rows, nb.ravel())), shape=(n, n))


def morans_i(values, weights, step: int = 48) -> np.ndarray:
    """Moran's I of every gene (column) on a row-normalized neighbour graph."""
    out = np.full(values.shape[1], np.nan)
    for g in range(0, values.shape[1], step):
        z = values[:, g:g + step].astype(np.float64)
        z -= z.mean(0)
        sd = z.std(0)
        z /= np.where(sd > 0, sd, 1.0)
        i = (z * (weights @ z)).sum(0) / len(z)
        i[sd == 0] = np.nan
        out[g:g + step] = i
    return out


# =============================================================================================
# Expression of single cells inside and between spots
# =============================================================================================
def pair_auc(expression, cell_type, cell_spot, max_cells: int = 30, seed: int = 0) -> float:
    """AUC with which the cosine distance between two cells of a spot separates pairs of
    different type from pairs of the same type, over the spots that hold several types."""
    rng = np.random.default_rng(seed)
    order = np.argsort(cell_spot, kind="stable")
    bounds = np.r_[np.unique(cell_spot[order], return_index=True)[1], len(order)]
    num = den = 0.0
    for a, b in zip(bounds[:-1], bounds[1:]):
        cells = order[a:b]
        if len(np.unique(cell_type[cells])) < 2:
            continue
        if len(cells) > max_cells:
            cells = rng.choice(cells, max_cells, replace=False)
        profile = unit(clr(expression[cells]))
        i, j = np.triu_indices(len(cells), 1)
        distance = (1.0 - profile @ profile.T)[i, j].astype(np.float32)
        different = cell_type[cells][i] != cell_type[cells][j]
        n1, n0 = different.sum(), (~different).sum()
        if n1 and n0:
            num += rankdata(distance)[different].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
    return num / den if den else float("nan")


def between_scores(p, t, group):
    """Correlation of the deviation from the mean of the cells that share a nearest spot,
    and median per-cell correlation, from log-normalized output ``p`` and truth ``t``."""
    pc, tc = p - p.mean(1, keepdims=True), t - t.mean(1, keepdims=True)
    den = np.sqrt((pc ** 2).sum(1) * (tc ** 2).sum(1))
    per_cell = np.where(den > 0, (pc * tc).sum(1) / np.where(den > 0, den, 1), np.nan)
    _, code, size = np.unique(group, return_inverse=True, return_counts=True)
    several = size[code] >= 2  # a lone cell has no deviation
    code = code[several]
    dp = (p[several] - group_mean(p[several], code, len(size))[code]).ravel().astype(np.float64)
    dt = (t[several] - group_mean(t[several], code, len(size))[code]).ravel().astype(np.float64)
    return pearson(dp, dt), float(np.nanmedian(per_cell))


def analyse_expression(sample, truth, methods, valid, gene: str | None = None,
                       seed: int = 0) -> None:
    # ---- one gene over the section, as measured and as returned by each method
    if PLOTS["dir"]:
        shown = scored_cells(sample, truth, valid)
        g = gene_index(sample.genes, gene, truth["counts"])
        maps = {name: (sample.cell_xy[shown], lognorm(x[shown])[:, g])
                for name, x in {"Ground Truth": truth["counts"], **methods}.items()}
        draw_maps("expression_gene_map", maps, f"{sample.genes[g]} (log CP10K)", vmin=0,
                  vmax=float(np.percentile(maps["Ground Truth"][1], 99.5)) or 1.0)

    # ---- cells inside spots.  The spot mean, which gives every cell the profile
    # of its spot, is the reference: 0 for the within-spot correlation and 0.5 for the AUC,
    # both by construction.
    inside = sample.inside
    for mask in valid:
        inside = inside & mask
    spot, types = sample.cell_spot[inside], truth["cell_type"][inside]
    measured = truth["counts"][inside]
    table = {}
    for name, x in {**methods, "Spot mean": sample.counts[sample.cell_spot]}.items():
        table[name] = {
            "within-spot r": within_spot_r(x[inside], measured, spot, sample.n_spots),
            "pair AUC": 0.5 if name == "Spot mean" else pair_auc(x[inside], types, spot,
                                                                 seed=seed),
            "relative distance": relative_distance(measured, x[inside])}
    truth_auc = pair_auc(measured, types, spot, seed=seed)  # the measured cells themselves
    report("expression", f"{int(inside.sum()):,} cells inside spots; pair AUC of the Ground "
                         f"Truth: {truth_auc:.4f}", table)
    draw_scores("expression_within_spot_r", column(table, "within-spot r"),
                "Within-spot r with the Ground Truth")
    draw_scores("expression_pair_auc", column(table, "pair AUC"),
                "AUC, different-type vs same-type pairs in a spot",
                reference={"Ground Truth": truth_auc})
    draw_scores("expression_relative_distance", column(table, "relative distance"),
                "Relative distance to the Ground Truth")

    # ---- cells between spots, which no spot measured.  Deviations are taken
    # from the mean of the cells that share a nearest spot; the two image-free references
    # are the profile of the nearest spot and the interpolation of the three nearest spots.
    between = scored_cells(sample, truth, valid) & ~sample.inside
    dist, near = cKDTree(sample.spot_xy).query(sample.cell_xy[between], k=3)
    weight = 1.0 / np.clip(dist, 1e-6, None)
    weight /= weight.sum(1, keepdims=True)
    spots = lognorm(sample.counts)
    outputs = {name: lognorm(x[between]) for name, x in methods.items()}
    outputs["Interpolation"] = sum(weight[:, k:k + 1] * spots[near[:, k]] for k in range(3))
    outputs["Nearest spot"] = spots[near[:, 0]]
    measured = lognorm(truth["counts"][between])
    table = {}
    for name, p in outputs.items():
        r, per_cell = between_scores(p, measured, near[:, 0])
        table[name] = {"between-spot r": r, "per-cell r": per_cell}
    report("expression", f"{int(between.sum()):,} cells between spots", table)
    draw_scores("expression_between_spot_r", column(table, "between-spot r"),
                "Between-spot r with the Ground Truth")
    draw_scores("expression_between_spot_per_cell_r", column(table, "per-cell r"),
                "Per-cell r with the Ground Truth")


# =============================================================================================
# Which cell is which inside a spot
# =============================================================================================
def spot_blocks(cell_spot, max_cells: int = 30, seed: int = 0) -> list:
    """Cells of every spot with at least two cells; large spots are subsampled."""
    rng = np.random.default_rng(seed)
    order = np.argsort(cell_spot, kind="stable")
    bounds = np.r_[np.unique(cell_spot[order], return_index=True)[1], len(order)]
    blocks = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        cells = order[a:b]
        if len(cells) >= 2:
            blocks.append(rng.choice(cells, max_cells, replace=False) if len(cells) > max_cells
                          else cells)
    return blocks


def identification(pred, truth, blocks):
    """Is the profile given to a cell closest to the measured profile of that same cell,
    among the cells of its spot?  Both are deviations from the spot mean.  Returns the rank
    of the own profile of every cell (0 = first, 0.5 = chance) and how often it is first,
    as a multiple of chance."""
    def log_centred(x):
        x = np.log(np.clip(np.asarray(x, np.float64), 1e-9, None))
        return (x - x.mean(1, keepdims=True)).astype(np.float32)

    x, t = log_centred(pred), log_centred(truth)
    ranks, first = [], 0.0
    for cells in blocks:
        a, b = x[cells], t[cells]
        similarity = unit(a - a.mean(0)) @ unit(b - b.mean(0)).T
        own = np.diag(similarity)[:, None]
        greater, ties = (similarity > own).sum(1), (similarity == own).sum(1) - 1
        ranks.append((greater + 0.5 * ties) / (len(cells) - 1))
        first += np.where(greater == 0, 1.0 / (1.0 + ties), 0.0).sum()
    ranks = np.concatenate(ranks)
    return ranks, (first / len(ranks)) / (len(blocks) / len(ranks))


def composition_error(types, true_types, cell_spot, n_spots: int, n_types: int) -> float:
    """Mean over spots, weighted by their number of cells, of the L1 distance between the
    predicted and the measured type composition (0 = identical, 2 = disjoint)."""
    def counts(t):
        h = np.zeros((n_spots, n_types))
        np.add.at(h, (cell_spot, t), 1.0)
        return h

    p, q = counts(true_types), counts(types)
    n = p.sum(1)
    ok = n > 0
    return float(np.average(np.abs(p[ok] - q[ok]).sum(1) / n[ok], weights=n[ok]))


def analyse_cell_types(sample, truth, methods, seed: int = 0) -> None:
    inside = sample.inside
    spot, true_types = sample.cell_spot[inside], truth["cell_type"][inside]
    measured = truth["counts"][inside]
    n_types = int(truth["cell_type"].max()) + 1
    centroids = type_centroids(measured, true_types, n_types)
    blocks = spot_blocks(spot, seed=seed)
    table, ranks, types = {}, {}, {}
    for name, x in methods.items():
        # ---- identification of a cell among the cells of its spot
        ranks[name], first = identification(x[inside], measured, blocks)
        # ---- cell-type composition of every spot, from the types read out
        types[name] = read_out_types(x[inside], centroids)
        error = composition_error(types[name], true_types, spot, sample.n_spots, n_types)
        table[name] = {"rank of own profile": float(ranks[name].mean()),
                       "ranked first, x chance": first, "composition error L1": error}
    # the read-out applied to the measured cells themselves bounds the composition error
    bound = composition_error(read_out_types(measured, centroids), true_types, spot,
                              sample.n_spots, n_types)
    report("cell_types", f"{int(inside.sum()):,} cells inside spots, {n_types} types; "
                         f"composition error of the Ground Truth read-out: {bound:.4f}", table)

    # ---- cell types of the cells inside spots, in one field of 600 um
    xy = sample.cell_xy[inside] * sample.px_um
    field = (np.abs(xy - np.median(xy, 0)) <= 300).all(1)
    draw_maps("cell_types_map",
              {"Ground Truth": (xy[field], true_types[field]),
               **{name: (xy[field], t[field]) for name, t in types.items()}},
              notes={name: f"{100 * (t[field] == true_types[field]).mean():.0f}% of cells correct"
                     for name, t in types.items()}, colours=(PALETTE * 3)[:n_types])
    if PLOTS["dir"]:
        with plot("cell_types_rank_of_own_profile") as (_, (ax,)):
            ax.hist(ranks["EMIT"], bins=25, density=True, color=COLORS["EMIT"], alpha=0.85)
            ax.axhline(1.0, color="#333333", ls="--", lw=0.8)  # chance
            ax.set_xlabel("Rank of the own profile of a cell (0 = first)")
            ax.set_ylabel("Density")
            ax.set_title("EMIT; dashed: chance")
    draw_scores("cell_types_ranked_first", column(table, "ranked first, x chance"),
                "Ranked first (x chance)", reference={"chance": 1.0})
    draw_scores("cell_types_composition_error", column(table, "composition error L1"),
                "Type-composition error per spot (L1)", reference={"Ground Truth": bound})


# =============================================================================================
# One map of all cells: tissue domains, cell types and spatially organized genes
# =============================================================================================
N_NEIGHBOURS, N_DOMAINS, N_MORAN, TOP_GENES = 20, 8, 8, 50


def neighbourhood_composition(nb, types, n_types: int) -> np.ndarray:
    """Type fractions among each cell and its neighbours."""
    n, k = nb.shape
    flat = np.repeat(np.arange(n), k) * n_types + types[nb].ravel()
    q = np.bincount(flat, minlength=n * n_types).reshape(n, n_types).astype(np.float32)
    q[np.arange(n), types] += 1
    return q / (k + 1)


def gene_correlation(a, b, step: int = 64) -> np.ndarray:
    """Pearson correlation across cells of every gene."""
    out = np.full(a.shape[1], np.nan)
    for g in range(0, a.shape[1], step):
        x = a[:, g:g + step].astype(np.float64)
        y = b[:, g:g + step].astype(np.float64)
        x -= x.mean(0)
        y -= y.mean(0)
        den = np.sqrt((x * x).sum(0) * (y * y).sum(0))
        out[g:g + step] = np.divide((x * y).sum(0), den, out=np.full(x.shape[1], np.nan),
                                    where=den > 0)
    return out


def analyse_all_cells(sample, truth, methods, valid) -> None:
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score

    # the cells inside spots (allocated) and between spots (predicted) are scored together
    cells = np.where(scored_cells(sample, truth, valid))[0]
    genes = np.where(measured_genes(sample.genes))[0]
    xy = sample.cell_xy[cells] * sample.px_um
    measured = truth["counts"][cells][:, genes]
    true_types = truth["cell_type"][cells]
    n_types = int(truth["cell_type"].max()) + 1
    centroids = type_centroids(measured, true_types, n_types)

    # measured tissue domains: k-means of the type composition of the 20 nearest cells
    nb, _ = neighbour_weights(xy, N_NEIGHBOURS)
    q_true = neighbourhood_composition(nb, true_types, n_types)
    fit = np.random.default_rng(0).choice(len(cells), min(len(cells), 200_000), replace=False)
    n_domains = min(N_DOMAINS, n_types)
    domains = KMeans(n_domains, n_init=10, random_state=0).fit(q_true[fit])
    true_domains = domains.predict(q_true)
    # measured spatial autocorrelation of every gene, on the 8 nearest cells
    _, weights = neighbour_weights(xy, N_MORAN)
    log_true = lognorm(measured)
    moran_true = morans_i(log_true, weights)
    top_true = set(np.argsort(-np.nan_to_num(moran_true, nan=-np.inf))[:TOP_GENES])

    table, domain_maps, gene_r, moran_rank = {}, {"Ground Truth": (xy, true_domains)}, {}, {}
    for name, x in {**methods, "Interpolation": interpolation(sample)}.items():
        x = x[cells][:, genes]
        types = read_out_types(x, centroids)
        # ---- domains, from the types of the output and the measured definition
        predicted = domains.predict(neighbourhood_composition(nb, types, n_types))
        domain_maps[name] = (xy, predicted)
        logx = lognorm(x)
        gene_r[name] = gene_correlation(logx, log_true)
        # ---- is the ranking of genes by Moran's I preserved?
        moran = morans_i(logx, weights)
        ok = np.isfinite(moran) & np.isfinite(moran_true)
        moran_rank[name] = (rankdata(-moran_true[ok]), rankdata(-moran[ok]))
        top = set(np.argsort(-np.nan_to_num(moran, nan=-np.inf))[:TOP_GENES])
        table[name] = {
            "domain ARI": adjusted_rand_score(true_domains, predicted),
            "type accuracy": float((types == true_types).mean()),
            "median gene r": float(np.nanmedian(gene_r[name])),
            "Moran rank rho": float(spearmanr(moran[ok], moran_true[ok])[0]),
            f"top-{TOP_GENES} genes found": len(top & top_true)}
    inside = sample.inside[cells]
    report("all_cells", f"{len(cells):,} cells ({int(inside.sum()):,} inside spots, "
                        f"{int((~inside).sum()):,} between), {len(genes)} genes", table)

    # ---- the domains of every cell
    notes = {name: f"ARI = {row['domain ARI']:.2f}" for name, row in table.items()}
    draw_maps("all_cells_domains", domain_maps,
              notes={"Ground Truth": f"{n_domains} domains", **notes}, colours=PALETTE[:n_domains])
    draw_scores("all_cells_domain_ari", column(table, "domain ARI"), "ARI of domains")
    draw_scores("all_cells_type_accuracy", column(table, "type accuracy"),
                "Accuracy of cell types")
    # ---- one point per gene, EMIT against each other output
    draw_scatters("all_cells_gene_r",
                  {name: (r, gene_r["EMIT"]) for name, r in gene_r.items() if name != "EMIT"},
                  "r of a gene, this output", "r of a gene, EMIT")
    draw_scatters("all_cells_moran_rank", moran_rank, "Rank by Moran's I, Ground Truth",
                  "Rank in the map of the output",
                  notes={name: f"rho = {row['Moran rank rho']:.2f}"
                         for name, row in table.items()})
    draw_scores("all_cells_spatial_genes_found", column(table, f"top-{TOP_GENES} genes found"),
                f"Ground Truth top-{TOP_GENES} genes in the top {TOP_GENES}")


# =============================================================================================
# A real Visium section: every cell mapped, and the measurement conserved
# =============================================================================================
def analyse_real_section(sample, methods, gene: str | None = None) -> None:
    # ---- the cells of each spot are summed and compared with the measurement of
    # that spot.  For EMIT the two are equal by construction, because the counts of the
    # cells inside a spot are an allocation of the spot counts.  Outputs of other methods
    # must be in counts for this comparison to be meaningful.
    table = {}
    for name, x in methods.items():
        error = conservation_error(x, sample.counts, sample.cell_spot)
        table[name] = {"median relative error": float(np.nanmedian(error)),
                       "largest relative error": float(np.nanmax(error))}
    report("real_section", f"{sample.n_cells:,} cells, {int(sample.inside.sum()):,} inside "
                           f"{sample.n_spots:,} spots, the others predicted", table)
    if not PLOTS["dir"]:
        return

    # ---- a gene as measured in the spots, and as returned for every cell
    spots = lognorm(sample.counts)
    g = gene_index(sample.genes, gene, sample.counts)
    maps = {"Visium spots": (sample.spot_xy, spots[:, g]),
            **{name: (sample.cell_xy, lognorm(x)[:, g]) for name, x in methods.items()}}
    draw_maps("real_section_gene_map", maps, f"{sample.genes[g]} (log CP10K)", vmin=0,
              vmax=float(np.percentile(spots[:, g], 99.5)) or 1.0,
              notes={"Visium spots": f"{sample.n_spots:,} spots",
                     **{name: f"{sample.n_cells:,} cells" for name in methods}})
    # ---- one point per spot and gene
    inside, scatters, rng = sample.inside, {}, np.random.default_rng(0)
    occupied = np.bincount(sample.cell_spot[inside], minlength=sample.n_spots) > 0
    for name, x in methods.items():
        total = np.zeros_like(sample.counts, dtype=np.float64)
        np.add.at(total, sample.cell_spot[inside], x[inside].astype(np.float64))
        measured, summed = sample.counts[occupied].ravel(), total[occupied].ravel()
        pick = rng.choice(len(measured), min(len(measured), 40_000), replace=False)
        scatters[name] = (np.log1p(measured[pick]), np.log1p(np.clip(summed[pick], 0, None)))
    draw_scatters("real_section_conservation", scatters, "Measured in the spot, log(1 + count)",
                  "The cells, summed, log(1 + count)")


# =============================================================================================
# Single cells from Visium HD bins
# =============================================================================================
BIN_UM, D_MAX_UM, RADIUS_UM = 2.0, 4.0, 10.0


def toy_scene(sigma_um: float = 0.0, width_um: float = 300.0, pitch_um: float = 10.0,
              n_types: int = 4, seed: int = 0) -> dict:
    """Synthetic tissue: nuclei of 3 um radius on a jittered hexagonal lattice of 10 um pitch
    (about 11,500 per mm2, a densely cellular section) emit transcripts around them;
    ``sigma_um`` moves every transcript before it is captured in a 2 um bin."""
    rng = np.random.default_rng(seed)
    rows = []
    for k, y in enumerate(np.arange(6.0, width_um - 6.0, pitch_um * np.sqrt(3) / 2)):
        x = np.arange(6.0 + (pitch_um / 2) * (k % 2), width_um - 6.0, pitch_um)
        rows.append(np.column_stack([x, np.full(len(x), y)]))
    cell_xy = np.concatenate(rows)
    cell_xy = cell_xy + rng.normal(0, 1.2, size=cell_xy.shape)
    cell_type = rng.integers(0, n_types, len(cell_xy))
    tx_cell = np.repeat(np.arange(len(cell_xy)), rng.poisson(220, len(cell_xy)))
    tx_xy = cell_xy[tx_cell] + rng.normal(0, 3.5, size=(len(tx_cell), 2))
    if sigma_um > 0:
        tx_xy = tx_xy + rng.normal(0, sigma_um, size=tx_xy.shape)
    tx_xy = np.clip(tx_xy, 0, width_um - 1e-6)

    n_side = int(np.ceil(width_um / BIN_UM))
    ii, jj = np.meshgrid(np.arange(n_side), np.arange(n_side))
    grid_xy = np.column_stack([(ii.ravel() + 0.5) * BIN_UM, (jj.ravel() + 0.5) * BIN_UM])
    dist, near = cKDTree(cell_xy).query(grid_xy)
    grid_nucleus = np.where(dist <= 3.0, near, -1)
    tx_grid = (tx_xy[:, 1] // BIN_UM).astype(int) * n_side + (tx_xy[:, 0] // BIN_UM).astype(int)
    occupied, tx_bin = np.unique(tx_grid, return_inverse=True)
    return {"bin_xy_um": grid_xy[occupied], "bin_nucleus": grid_nucleus[occupied],
            "ref_xy_um": grid_xy[grid_nucleus >= 0], "ref_nucleus": grid_nucleus[grid_nucleus >= 0],
            "cell_xy_um": cell_xy, "cell_type": cell_type, "tx_bin": tx_bin, "tx_cell": tx_cell,
            "scored": ((cell_xy > 25) & (cell_xy < width_um - 25)).all(1)}


def bin_memberships(scene) -> dict:
    """Cells x bins membership under three rules."""
    n_cells, n_bins = len(scene["cell_xy_um"]), len(scene["bin_xy_um"])

    def exclusive(owner):
        b = np.where(owner >= 0)[0]
        return sparse.csr_matrix((np.ones(len(b), np.float32), (owner[b], b)),
                                 shape=(n_cells, n_bins))

    # EMIT-HD: a bin inside a nucleus belongs to that cell, any other bin to the nearest
    # nucleus within 4 um, so every bin has at most one owner
    owner = assign_bins(scene["bin_xy_um"], scene["bin_nucleus"], d_max_um=D_MAX_UM,
                        ref_xy_um=scene["ref_xy_um"], ref_nucleus=scene["ref_nucleus"])
    # nucleus-centred radius (the rule of Thor's HD2cell): every bin within 10 um of a
    # nucleus centroid, counted for every nucleus near it
    near = cKDTree(scene["bin_xy_um"]).query_ball_point(scene["cell_xy_um"], RADIUS_UM)
    rows = np.repeat(np.arange(n_cells), [len(v) for v in near])
    radius = sparse.csr_matrix((np.ones(len(rows), np.float32),
                                (rows, np.concatenate([np.asarray(v, np.int64) for v in near]))),
                               shape=(n_cells, n_bins))
    # oracle: each bin to the cell that owns most of its transcripts (nucleus bins stay with
    # their nucleus), the ceiling of any rule that moves whole bins
    owned = scene["tx_cell"] >= 0
    key, count = np.unique(scene["tx_bin"][owned] * (n_cells + 1) + scene["tx_cell"][owned],
                           return_counts=True)
    b, cell = key // (n_cells + 1), key % (n_cells + 1)
    order = np.lexsort((-count, b))
    first = np.r_[True, b[order][1:] != b[order][:-1]]
    oracle = np.full(n_bins, -1, np.int64)
    oracle[b[order][first]] = cell[order][first]
    oracle[scene["bin_nucleus"] >= 0] = scene["bin_nucleus"][scene["bin_nucleus"] >= 0]
    return {"EMIT-HD": exclusive(owner), "Radius 10 um": radius, "Oracle": exclusive(oracle)}


def score_assignment(scene, membership) -> dict:
    """Of the transcripts given to a cell, how many are its own (precision); of its own
    transcripts, how many it receives (recall); and the share from a cell of another type."""
    n_cells, n_bins = membership.shape
    tx_bin, tx_cell, cell_type = scene["tx_bin"], scene["tx_cell"], scene["cell_type"]
    scored = scene.get("scored")
    scored = np.ones(n_cells, bool) if scored is None else np.asarray(scored, bool)
    owned = tx_cell >= 0
    assigned = membership @ np.bincount(tx_bin, minlength=n_bins).astype(float)
    truth = np.bincount(tx_cell[owned], minlength=n_cells).astype(float)
    pairs = membership.tocoo()
    held = np.sort(pairs.row.astype(np.int64) * n_bins + pairs.col)
    hit = np.isin(tx_cell[owned] * n_bins + tx_bin[owned], held)
    correct = np.bincount(tx_cell[owned][hit], minlength=n_cells).astype(float)
    tx_type = scene.get("tx_type")  # type of the owner of each transcript, -1 if unknown
    if tx_type is None:
        tx_type = np.where(owned, cell_type[np.clip(tx_cell, 0, None)], -1)
    typed = np.asarray(tx_type) >= 0
    by_type = sparse.csr_matrix((np.ones(int(typed.sum())), (tx_bin[typed], tx_type[typed])),
                                shape=(n_bins, int(cell_type.max()) + 1))
    received = np.asarray((membership @ by_type).todense())
    own_type = received[np.arange(n_cells), np.clip(cell_type, 0, None)]
    precision = correct[scored].sum() / max(assigned[scored].sum(), 1)
    recall = correct[scored].sum() / max(truth[scored].sum(), 1)
    return {"precision": precision, "recall": recall,
            "F1": 2 * precision * recall / max(precision + recall, 1e-12),
            "other-type share": (received.sum(1) - own_type)[scored].sum()
            / max(received[scored].sum(), 1)}


def draw_bins(name: str, scene, membership) -> None:
    """An 80 um field: the owner of every bin, and the number of cells a bin is counted for."""
    xy, cells = scene["bin_xy_um"], scene["cell_xy_um"]
    centre = np.median(cells, 0)
    field = (np.abs(xy - centre) <= 40).all(1)
    nuclei = cells[(np.abs(cells - centre) <= 40).all(1)]
    one = membership["EMIT-HD"].tocoo()
    owner = np.full(len(xy), -1, np.int64)
    owner[one.col] = one.row
    shade = np.where(owner[field] >= 0, owner[field] % len(PALETTE), len(PALETTE))
    per_bin = np.asarray(membership["Radius 10 um"].sum(0)).ravel()[field]
    with plot(name, ncols=2, width=2.4, height=2.3) as (fig, (left, right)):
        left.scatter(xy[field, 0], xy[field, 1], c=np.array(PALETTE + ["#dddddd"])[shade], s=8,
                     marker="s", linewidths=0)
        left.set_title("EMIT-HD: one cell per bin")
        image = right.scatter(xy[field, 0], xy[field, 1], c=np.clip(per_bin, 0, 3), s=8,
                              marker="s", linewidths=0, cmap="Oranges", vmin=0, vmax=3)
        right.set_title("Radius 10 um: cells per bin")
        fig.colorbar(image, ax=right, ticks=[0, 1, 2, 3], shrink=0.7)
        for ax in (left, right):
            ax.scatter(nuclei[:, 0], nuclei[:, 1], s=3, color="#222222", linewidths=0)
            ax.set_aspect("equal")
            ax.invert_yaxis()
            ax.set_xticks([])
            ax.set_yticks([])


def analyse_visium_hd(scene_files, seed: int = 0) -> None:
    if scene_files:
        scenes = {}
        for path in scene_files:
            with np.load(path, allow_pickle=True) as z:
                scenes[os.path.splitext(os.path.basename(path))[0]] = {k: z[k] for k in z.files}
        displacement = None
    else:
        # ---- capture is not in place; transcripts are displaced before binning
        displacement = (0.0, 2.0, 4.0)
        scenes = {f"synthetic tissue, displacement {s:g} um": toy_scene(s, seed=seed)
                  for s in displacement}
    tables = []
    for k, (name, scene) in enumerate(scenes.items()):
        # ---- bins given back to cells under each rule
        membership = bin_memberships(scene)
        table = {rule: score_assignment(scene, m) for rule, m in membership.items()}
        tables.append(table)
        report("visium_hd", f"{name}: {len(scene['bin_xy_um']):,} bins with transcripts, "
                            f"{len(scene['cell_xy_um']):,} cells", table)
        if not PLOTS["dir"] or (displacement and k):  # synthetic tissue: drawn undisplaced
            continue
        tag = "" if displacement else f"_{name}"
        # ---- which cell a bin belongs to
        draw_bins(f"visium_hd_bins{tag}", scene, membership)
        draw_scores(f"visium_hd_bin_to_cell_f1{tag}", column(table, "F1"), "Bin-to-cell F1")
        draw_scores(f"visium_hd_other_type_share{tag}", column(table, "other-type share"),
                    "Share of transcripts from another cell type")
    if PLOTS["dir"] and displacement:
        with plot("visium_hd_capture_displacement", width=3.0) as (_, (ax,)):
            for rule in tables[0]:
                ax.plot(displacement, [t[rule]["F1"] for t in tables], marker="o", ms=4,
                        color=COLORS[rule], ls=":" if rule == "Oracle" else "-", label=rule)
            ax.set_xticks(displacement)
            ax.set_ylim(0, 1)
            ax.set_xlabel("Transcript displacement before capture (um)")
            ax.set_ylabel("Bin-to-cell F1")
            ax.legend(frameon=False)


# =============================================================================================
# Differential expression where two cell types share a spot, and gene programmes
# =============================================================================================
MIN_CELLS, MIN_SPOTS = 2, 50          # cells of each type per spot; spots per type pair
N_PROGRAMME_GENES, N_PROGRAMMES, MIN_PROGRAMME = 100, 6, 3


def interface_de(sample, truth, methods, min_spots: int = MIN_SPOTS):
    """Within-spot differential expression between two cell types, per type pair.

    For two cell types A and B, the spots that hold at least two cells of each are taken,
    and the difference between the mean expression of the A cells and of the B cells of the
    same spot is averaged over those spots.  A copy of the spot profile gives exactly zero.
    Returns the type pairs and ``{name: (pairs, genes) differences}``, the truth included.
    """
    keep = sample.inside & (truth["counts"].sum(1) >= MIN_COUNTS)
    genes = measured_genes(sample.genes)
    spot, types = sample.cell_spot[keep], truth["cell_type"][keep]
    n_types = int(truth["cell_type"].max()) + 1
    key = spot * n_types + types
    held = np.bincount(key, minlength=sample.n_spots * n_types).reshape(sample.n_spots, n_types)
    pairs = [(a, b, np.where((held[:, a] >= MIN_CELLS) & (held[:, b] >= MIN_CELLS))[0])
             for a in range(n_types) for b in range(a + 1, n_types)]
    pairs = [p for p in pairs if len(p[2]) >= min_spots]
    to_groups = sparse.csr_matrix((np.ones(len(key), np.float32), (key, np.arange(len(key)))),
                                  shape=(sample.n_spots * n_types, len(key)))
    size = np.clip(np.bincount(key, minlength=sample.n_spots * n_types), 1, None)[:, None]

    def difference(x):
        mean = np.asarray(to_groups @ lognorm(x[keep])[:, genes], np.float64) / size
        mean = mean.reshape(sample.n_spots, n_types, -1)
        return np.array([(mean[q, a] - mean[q, b]).mean(0) for a, b, q in pairs])

    return pairs, {name: difference(x) for name, x in {"Ground Truth": truth["counts"],
                                                        **methods}.items()}


def gene_programmes(smoothed) -> np.ndarray:
    """Group genes by average-linkage clustering of one minus their correlation;
    programme 0 is the largest."""
    with np.errstate(invalid="ignore", divide="ignore"):  # a constant gene has no correlation
        r = np.nan_to_num(np.corrcoef(smoothed.T))
    np.fill_diagonal(r, 1.0)
    d = np.clip(1.0 - 0.5 * (r + r.T), 0.0, 2.0)
    np.fill_diagonal(d, 0.0)
    label = fcluster(linkage(squareform(d, checks=False), "average"), N_PROGRAMMES, "maxclust")
    order = np.argsort(-np.bincount(label, minlength=label.max() + 1)[1:], kind="stable")
    rank = np.empty(len(order), int)
    rank[order] = np.arange(len(order))
    return rank[label - 1]


def programme_activity(sample, truth, methods, valid):
    """Activity of the measured gene programmes in the truth and in every output.

    On the map of all cells, the 100 genes with the highest measured Moran's I are grouped
    into programmes from their expression averaged over a cell and its 8 nearest cells; the
    activity of a programme in a cell is the mean standardized expression of its genes.
    Returns the cell positions and ``{name: (cells, programmes) activity}``.
    """
    cells = np.where(scored_cells(sample, truth, valid))[0]
    genes = np.where(measured_genes(sample.genes))[0]
    xy = sample.cell_xy[cells] * sample.px_um
    _, weights = neighbour_weights(xy, N_MORAN)
    smooth = (sparse.identity(len(cells), format="csr") + weights * N_MORAN) / (N_MORAN + 1.0)
    log_true = lognorm(truth["counts"][cells][:, genes])
    moran = np.nan_to_num(morans_i(log_true, weights), nan=-np.inf)
    top = np.argsort(-moran)[:N_PROGRAMME_GENES]
    top = top[moran[top] > 0]
    label = gene_programmes(smooth @ log_true[:, top].astype(np.float64))
    used = [k for k in range(N_PROGRAMMES) if (label == k).sum() >= MIN_PROGRAMME]

    def activity(x):
        z = lognorm(x[cells][:, genes])[:, top].astype(np.float64)
        z -= z.mean(0)
        sd = z.std(0)
        z /= np.where(sd > 0, sd, 1.0)
        return np.stack([z[:, label == k].mean(1) for k in used], 1)

    return xy, {name: activity(x) for name, x in {"Ground Truth": truth["counts"],
                                                   **methods}.items()}


def analyse_interface(sample, truth, methods, valid) -> None:
    # ---- interface differential expression, scored by its correlation with
    # the measured difference across genes (median over type pairs)
    pairs, difference = interface_de(sample, truth, methods,
                                     min_spots=10 if sample.name == "toy" else MIN_SPOTS)
    if not pairs:
        raise SystemExit("no pair of cell types shares enough spots")
    r_de = {name: [pearson(d, t) for d, t in zip(difference[name], difference["Ground Truth"])]
            for name in methods}
    # ---- spatial gene programmes, scored by the correlation of their activity
    # with the measured activity across cells (median over programmes)
    xy, activity = programme_activity(sample, truth,
                                      {**methods, "Interpolation": interpolation(sample)}, valid)
    measured = activity.pop("Ground Truth")
    r_programme = {name: [pearson(a[:, j], measured[:, j]) for j in range(measured.shape[1])]
                   for name, a in activity.items()}
    table = {}
    for name in activity:  # the interpolation repeats the spot profile inside a spot
        table[name] = {
            "interface DE r": float(np.median(r_de[name])) if name in r_de else 0.0,
            "programme activity r": float(np.median(r_programme[name]))}
    report("interface", f"{len(pairs)} type pairs that share spots, {measured.shape[1]} "
                        f"programmes", table)

    # ---- one point per gene, for the pair of types that differ most in the
    # measurement (the pair is chosen from the truth alone)
    k = int(np.argmax(np.linalg.norm(difference["Ground Truth"], axis=1)))
    draw_scatters("interface_de_scatter",
                  {name: (difference["Ground Truth"][k], difference[name][k]) for name in methods},
                  "Ground Truth", f"Type {pairs[k][0] + 1} - type {pairs[k][1] + 1}, within spots",
                  diagonal=False, notes={name: f"r = {r_de[name][k]:.2f}" for name in methods})
    draw_scores("interface_de_r", column(table, "interface DE r", methods),
                "Interface DE agreement with the Ground Truth (r)")
    # ---- activity of the largest programme in every cell
    vmax = float(np.percentile(np.abs(measured[:, 0]), 99))
    draw_maps("programme_activity_map",
              {"Ground Truth": (xy, measured[:, 0]),
               **{name: (xy, a[:, 0]) for name, a in activity.items()}},
              "Activity of programme 1", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
              notes={name: f"r = {r[0]:.2f}" for name, r in r_programme.items()})
    draw_scores("programme_activity_r", column(table, "programme activity r"),
                "r of programme activity")


# =============================================================================================
ANALYSES = {
    "expression": "Expression of single cells inside and between spots",
    "cell_types": "Which cell is which inside a spot",
    "all_cells": "One map of all cells: tissue domains, cell types and spatially organized genes",
    "real_section": "A real Visium section: every cell mapped, and the measurement conserved",
    "visium_hd": "Single cells from Visium HD bins",
    "interface": "Differential expression where two cell types share a spot, and gene programmes",
}


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    p.add_argument("--analyses", nargs="*", default=list(ANALYSES), choices=list(ANALYSES),
                   metavar="NAME", help="which analyses to run: " + ", ".join(ANALYSES))
    p.add_argument("--section", default="toy",
                   help="benchmark section (.npz) for expression, cell_types, all_cells, interface")
    p.add_argument("--features", help="image embeddings of its cells (.npz from `emit features`)")
    p.add_argument("--compare", nargs="*", default=[], metavar="NAME=FILE",
                   help="per-cell output of another method on the benchmark section")
    p.add_argument("--gene", help="gene drawn in the maps (default: the most variable gene)")
    p.add_argument("--spaceranger",
                   help="real_section: Space Ranger directory of a real Visium section")
    p.add_argument("--nuclei", help="real_section: its nuclei (.npz from `emit segment`)")
    p.add_argument("--scene", nargs="*", default=[], help="visium_hd: tissue windows (.npz)")
    p.add_argument("--out", default="reproduce_out",
                   help="directory for the plots, the values and the cached fits")
    p.add_argument("--no-plots", action="store_true", help="compute and print only")
    p.add_argument("--show", action="store_true",
                   help="keep the plots open, so that a notebook displays them")
    p.add_argument("--rounds", type=int, default=None, help="EM rounds (default 20; toy 8)")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default=None)
    args = p.parse_args(argv)
    RESULTS.clear()
    PLOTS.update(dir=None if args.no_plots else args.out, show=args.show)
    if PLOTS["dir"]:
        try:
            import matplotlib

            if "IPython" not in sys.modules:  # a plain run needs no window
                matplotlib.use("Agg")
        except ImportError:
            print("matplotlib is not installed: the values are computed, nothing is drawn")
            PLOTS["dir"] = None

    wanted = [name for name in ANALYSES if name in args.analyses]
    if set(wanted) & {"expression", "cell_types", "all_cells", "interface"}:
        # the benchmark section and the outputs to compare
        sample, truth = load_section(args.section, args.features)
        compared = load_compared(args.compare, sample)
        methods = {"EMIT": emit_output(sample, args), **{k: v[0] for k, v in compared.items()}}
        valid = [v[1] for v in compared.values()]
        print(f"\nBenchmark section {sample.name}: {sample.n_cells:,} cells, {sample.n_spots:,} "
              f"simulated spots, {sample.n_genes:,} genes; methods: {', '.join(methods)}")
    for name in wanted:
        print(f"\n{ANALYSES[name]}")
        if name == "expression":
            analyse_expression(sample, truth, methods, valid, args.gene, args.seed)
        elif name == "cell_types":
            analyse_cell_types(sample, truth, methods, args.seed)
        elif name == "all_cells":
            analyse_all_cells(sample, truth, methods, valid)
        elif name == "real_section":
            if args.spaceranger:
                emit.setup_logging()
                tag = os.path.basename(os.path.normpath(args.spaceranger))
                real = emit.build_sample(emit.read_visium(args.spaceranger),
                                         emit.read_nuclei(args.nuclei), name=tag)
            else:  # the synthetic section stands in for a real one; its truth is not read
                real, _ = load_section("toy")
            analyse_real_section(real, {"EMIT": emit_output(real, args)}, args.gene)
        elif name == "visium_hd":
            analyse_visium_hd(args.scene, args.seed)
        elif name == "interface":
            analyse_interface(sample, truth, methods, valid)

    import pandas as pd

    results = pd.DataFrame(RESULTS)
    os.makedirs(args.out, exist_ok=True)
    results.to_csv(os.path.join(args.out, "analyses.csv"), index=False)
    print(f"\nvalues -> {os.path.join(args.out, 'analyses.csv')}")
    return results


if __name__ == "__main__":
    main()
